# 3D PCA - Index

In [ ]:
from __future__ import annotations

import ast
import random
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from sklearn.decomposition import PCA
from transformers import AutoConfig, AutoModel, AutoTokenizer

import plotly.graph_objects as go


In [ ]:
RAND_SEED = 42
random.seed(RAND_SEED)
np.random.seed(RAND_SEED)
torch.manual_seed(RAND_SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)

In [ ]:
DATA_FILE = Path("data/en_ewt-ud-train_sentences.csv")
if not DATA_FILE.exists():
    DATA_FILE = Path("code/data/en_ewt-ud-train_sentences.csv")
CSV_PATH = str(DATA_FILE)
TOKENS_COL = "tokens"
INDEX_COL  = "index"

MODEL_NAME = "gpt2"          # or "bert-base-uncased"
WORD_REP_MODE = "last"       # {"first","last","mean"} for word-level pooling over subtokens
MAX_LENGTH = 256
BATCH_SIZE_SENT = 8

INDEX_MAX_CLASS = 10
INCLUDE_ZERO_CLASS = True

MAX_POINTS_PER_CLASS = 2500   # None for all 
LAYERS_TO_PLOT = [1, 12]      # 


In [ ]:
def _to_list(x):
    if isinstance(x, list):
        return x
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return []
    if isinstance(x, str):
        s = x.strip()
        if not s:
            return []
        try:
            v = ast.literal_eval(s)
            return v if isinstance(v, list) else [v]
        except Exception:
            return s.split()
    return list(x)

def load_sentence_df(csv_path: str) -> pd.DataFrame:
    df = pd.read_csv(csv_path)
    for col in ["sentence_id", TOKENS_COL, INDEX_COL]:
        if col not in df.columns:
            raise ValueError(f"CSV must contain column: {col}")
    df = df.copy()
    df["sentence_id"] = df["sentence_id"].astype(str)
    df[TOKENS_COL] = df[TOKENS_COL].apply(_to_list)
    df[INDEX_COL] = df[INDEX_COL].apply(_to_list)
    return df

def bucket_index(i: int, index_max: int) -> int:
    return int(min(int(i), int(index_max)))

In [ ]:
def explode_index(df_sent: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for _, row in df_sent.iterrows():
        sid = str(row["sentence_id"])
        toks = row[TOKENS_COL]
        idxs = row[INDEX_COL]
        n = min(len(toks), len(idxs))
        for wid in range(n):
            idx_val = idxs[wid]
            if idx_val is None:
                continue
            try:
                idx_int = int(idx_val)
            except Exception:
                continue
            if (not INCLUDE_ZERO_CLASS) and idx_int == 0:
                continue
            rows.append({
                "sentence_id": sid,
                "word_id": wid,
                "index_raw": idx_int,
                "index_class": bucket_index(idx_int, INDEX_MAX_CLASS),
            })
    return pd.DataFrame(rows)


In [ ]:
df_sent = load_sentence_df(CSV_PATH)
idx_df = explode_index(df_sent)

print("Sentences:", len(df_sent))
print("Token rows:", len(idx_df))
idx_df["index_class"].value_counts().sort_index()

In [ ]:
def build_tokenizer_and_model(model_name: str):
    tok = AutoTokenizer.from_pretrained(model_name, use_fast=True, add_prefix_space=True)
    if tok.pad_token is None:
        if tok.eos_token is None:
            raise ValueError(f"Tokenizer for {model_name} has no pad_token and no eos_token; set a pad token manually.")
        tok.pad_token = tok.eos_token
    if not getattr(tok, "is_fast", False):
        raise ValueError(
            f"Tokenizer for {model_name} is not a fast tokenizer; word alignment requires a fast tokenizer."
        )
    cfg = AutoConfig.from_pretrained(model_name, output_hidden_states=True)
    model = AutoModel.from_pretrained(model_name, config=cfg)
    model.eval().to(DEVICE)
    return tok, model

def sample_per_class(df: pd.DataFrame, class_col: str, max_per_class: int | None) -> pd.DataFrame:
    parts = []
    for c, g in df.groupby(class_col):
        if max_per_class is not None and len(g) > max_per_class:
            g = g.sample(n=max_per_class, random_state=RAND_SEED)
        parts.append(g)
    return pd.concat(parts, axis=0).reset_index(drop=True)

In [ ]:
@torch.no_grad()
def embed_subset_layers(
    df_sent: pd.DataFrame,
    subset_df: pd.DataFrame,
    tokenizer,
    model,
    layers: List[int],
    word_rep_mode: str,
    batch_size: int = 8,
    max_length: int = 256,
) -> Tuple[Dict[int, np.ndarray], np.ndarray]:
    """
    Returns:
      X_by_layer[layer] = (N,D)
      y_class = (N,) int index_class
    """
    subset_df = subset_df.copy()
    subset_df["sentence_id"] = subset_df["sentence_id"].astype(str)

    by_sid: Dict[str, List[Tuple[int, int, int]]] = {}
    for gidx, (sid, wid, c) in enumerate(subset_df[["sentence_id","word_id","index_class"]].itertuples(index=False, name=None)):
        by_sid.setdefault(str(sid), []).append((gidx, int(wid), int(c)))

    sids = list(by_sid.keys())
    df_sel = (
        df_sent[df_sent["sentence_id"].isin(sids)]
        .drop_duplicates("sentence_id")
        .set_index("sentence_id")
        .loc[sids]
    )

    N = len(subset_df)
    X_by_layer = {}
    y = np.zeros(N, dtype=np.int32)
    filled = np.zeros(N, dtype=bool)

    enc_kwargs = dict(
        is_split_into_words=True,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=max_length,
    )

    for start in tqdm(range(0, len(sids), batch_size), desc="Embedding"):
        batch_ids = sids[start:start + batch_size]
        toks_batch = df_sel.loc[batch_ids, TOKENS_COL].tolist()

        enc = tokenizer(toks_batch, **enc_kwargs)
        word_ids_list = [enc.word_ids(batch_index=bi) for bi in range(len(batch_ids))]

        enc = enc.to(DEVICE)
        out = model(**enc)
        hs = out.hidden_states  

        if not X_by_layer:
            D = hs[0].shape[-1]
            for l in layers:
                X_by_layer[l] = np.zeros((N, D), dtype=np.float32)

        for bi, sid in enumerate(batch_ids):
            word_ids = word_ids_list[bi]
            pos_map: Dict[int, List[int]] = {}
            for ti, wid in enumerate(word_ids):
                if wid is not None:
                    pos_map.setdefault(int(wid), []).append(int(ti))

            for gidx, wid, c in by_sid[str(sid)]:
                if wid not in pos_map:
                    continue
                positions = pos_map[wid]

                if word_rep_mode == "first":
                    tok_pos = positions[0]
                    for l in layers:
                        X_by_layer[l][gidx, :] = hs[l][bi, tok_pos, :].detach().float().cpu().numpy()

                elif word_rep_mode == "last":
                    tok_pos = positions[-1]
                    for l in layers:
                        X_by_layer[l][gidx, :] = hs[l][bi, tok_pos, :].detach().float().cpu().numpy()

                elif word_rep_mode == "mean":
                    idx_t = torch.tensor(positions, device=hs[0].device)
                    for l in layers:
                        X_by_layer[l][gidx, :] = hs[l][bi, idx_t, :].mean(dim=0).detach().float().cpu().numpy()

                else:
                    raise ValueError("word_rep_mode must be one of {'first','last','mean'}")

                y[gidx] = c
                filled[gidx] = True

        del out, hs
        if DEVICE == "cuda":
            torch.cuda.empty_cache()

    if not filled.all():
        print(f"Dropped {(~filled).sum()} words (alignment / truncation).")
    X_by_layer = {l: X[filled] for l, X in X_by_layer.items()}
    return X_by_layer, y[filled]

In [ ]:
def plot_pca3d(X: np.ndarray, y: np.ndarray, title: str) -> go.Figure:
    pca = PCA(n_components=3, random_state=RAND_SEED)
    Z = pca.fit_transform(X)

    classes = np.unique(y)
    base_colors = [
        "#1f77b4","#ff7f0e","#2ca02c","#d62728","#9467bd","#8c564b",
        "#e377c2","#7f7f7f","#bcbd22","#17becf",
    ]

    fig = go.Figure()
    for i, c in enumerate(sorted(classes.tolist())):
        mask = (y == c)
        fig.add_trace(go.Scatter3d(
            x=Z[mask, 0],
            y=Z[mask, 1],
            z=Z[mask, 2],
            mode="markers",
            name=f"index={c}" + (f"+" if c == INDEX_MAX_CLASS else ""),
            marker=dict(size=2, opacity=0.65, color=base_colors[i % len(base_colors)]),
        ))

    fig.update_layout(
        title=title,
        scene=dict(
            xaxis_title="PC1",
            yaxis_title="PC2",
            zaxis_title="PC3",
        ),
        legend=dict(itemsizing="constant"),
        margin=dict(l=0, r=0, t=40, b=0),
        height=650,
    )
    return fig


In [ ]:
subset = sample_per_class(idx_df, "index_class", MAX_POINTS_PER_CLASS)
print("Total plotted points:", len(subset))

tokenizer, model = build_tokenizer_and_model(MODEL_NAME)
X_by_layer, y = embed_subset_layers(
    df_sent=df_sent,
    subset_df=subset[["sentence_id","word_id","index_class"]],
    tokenizer=tokenizer,
    model=model,
    layers=LAYERS_TO_PLOT,
    word_rep_mode=WORD_REP_MODE,
    batch_size=BATCH_SIZE_SENT,
    max_length=MAX_LENGTH,
)

for layer in LAYERS_TO_PLOT:
    fig = plot_pca3d(X_by_layer[layer], y, title=f"{MODEL_NAME} — layer {layer} — PCA3D by index class")
    fig.show()